# Exercices : Charger et explorer un jeu de données

Pour chaque exercice, complète les cellules marquées `# TODO`, exécute-les et compare avec le résultat attendu écrit en commentaire. Ne regarde la solution qu'après avoir vraiment essayé !

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/surybang/python_data/main/data/ventes.csv"

---

## 1. Charger le fichier des ventes

Charge `ventes.csv` dans un DataFrame `ventes`, en lisant la colonne `date_vente` comme une date. Affiche ensuite sa taille et le type de `date_vente`.

In [ ]:
# TODO
ventes = ...

print(ventes.shape)                 # attendu : (304, 9)
print(ventes["date_vente"].dtype)   # attendu : datetime64[us]

<details>
<summary>💡 Solution</summary>

```python
ventes = pd.read_csv(URL, parse_dates=["date_vente"])

print(ventes.shape)
print(ventes["date_vente"].dtype)
```
</details>

---

## 2. Un export Excel « à la française »

Le service logistique t'envoie l'état de ses stocks d'emballages, exporté d'Excel. Le contenu du fichier est copié dans la variable `export` ci-dessous. `io.StringIO` permet de le lire **comme si c'était un fichier**.

Charge-le correctement, puis calcule la valeur totale du stock (stock multiplié par prix, puis somme).

In [ ]:
import io

export = """reference;libelle;stock;prix
P01;Palette bois;40;12,5
P02;Film étirable;120;8,9
P03;Carton 60x40;300;1,2
"""

# TODO : complète la lecture
stocks = pd.read_csv(io.StringIO(export))

print(stocks.shape)                           # attendu : (3, 4)
print(stocks["prix"].dtype)                   # attendu : float64
print((stocks["stock"] * stocks["prix"]).sum())   # attendu : 1928.0

<details>
<summary>💡 Solution</summary>

```python
stocks = pd.read_csv(io.StringIO(export), sep=";", decimal=",")
```
</details>

🤔 Essaie avec `sep=";"` mais **sans** `decimal=","` : quel est le type de `prix` ? Que devient le calcul de la valeur du stock ?

---

## 3. Devine sans exécuter

Sans exécuter la cellule, devine ce qui va se passer. Écris ta réponse en commentaire, puis exécute pour vérifier.

In [ ]:
# Ta prédiction :
#

ventes.shape()

<details>
<summary>💡 Solution</summary>

`TypeError: 'tuple' object is not callable`

`shape` est un **attribut** (une information stockée, ici un tuple `(304, 9)`), pas une méthode : il s'écrit sans parenthèses. Avec des parenthèses, Python essaie « d'appeler » le tuple comme une fonction.
</details>

---

## 4. Réécrire en chaîne

Le code ci-dessous fonctionne, mais il crée deux variables intermédiaires inutiles. Réécris-le en **une seule chaîne** entre parenthèses, une étape par ligne.

In [ ]:
raw = pd.read_csv(URL, parse_dates=["date_vente"])
drop_cols = raw.drop(columns=["categorie", "id_client"])
tail = drop_cols.tail(5)
tail

In [ ]:
# TODO
tail = (
    ...
)
tail

# attendu :
#      id_vente date_vente produit     ville  quantite  prix_unitaire  statut
# 302      1210 2026-06-30  Casque    Nantes         2           79.0  LIVREE
# 303      1220 2026-06-30   Écran  Bordeaux         5          189.0  LIVREE

<details>
<summary>💡 Solution</summary>

```python
dernieres = (
    pd.read_csv(URL, parse_dates=["date_vente"])
    .drop(columns=["categorie", "id_client"])
    .tail(2)
)
dernieres
```
</details>

---

## 5. Montant et montant TTC

Le prix unitaire est hors taxes. Ajoute à `ventes`, avec **un seul** `assign` et `pd.col` :

- une colonne `montant` : quantité fois prix unitaire ;
- une colonne `montant_ttc` : le montant multiplié par 1,2.

In [ ]:
# TODO
ventes = (
    ventes
    .assign(...)
)

print(ventes[["montant", "montant_ttc"]].head(3))
# attendu :
#    montant  montant_ttc
# 0    -39.8       -47.76
# 1    158.0       189.60
# 2   1192.0      1430.40

print(ventes["montant"].max())   # attendu : 1992.0

<details>
<summary>💡 Solution</summary>

```python
ventes = (
    ventes
    .assign(
        montant=pd.col("quantite") * pd.col("prix_unitaire"),
        montant_ttc=pd.col("montant") * 1.2,
    )
)
```
</details>

🤔 `montant_ttc` utilise `montant`, créée **dans le même** `assign`. Ça marche parce que les colonnes sont créées dans l'ordre où tu les écris. Que se passe-t-il si tu inverses les deux lignes ?

---

## 6. Enquête sur le fichier

Réponds aux questions avec une ligne de code chacune.

In [ ]:
# TODO : combien de produits différents sont vendus ?
...    # attendu : 8

In [ ]:
# TODO : combien de clients distincts ont acheté ?
...    # attendu : 60

In [ ]:
# TODO : quelles sont les 3 villes les plus fréquentes, en comptant aussi les villes manquantes ?
...
# attendu :
# ville
# Paris        85
# Lyon         69
# Marseille    43
# Name: count, dtype: int64

<details>
<summary>💡 Solution</summary>

```python
ventes["produit"].nunique()
ventes["id_client"].nunique()
ventes["ville"].value_counts(dropna=False).head(3)
```
</details>

---

## 7. Chasse au bug

Ce code contient **trois** erreurs. L'une d'elles ne lève même pas d'erreur, mais n'affiche pas ce qu'on attend. Corrige-les.

In [ ]:
# TODO : corrige les trois erreurs
ventes_bug = pd.read_csv(URL, parse_date=["date_vente"])
ventes_bug.info
ventes_bug["Statut"].value_counts()

<details>
<summary>💡 Solution</summary>

```python
ventes_bug = pd.read_csv(URL, parse_dates=["date_vente"])   # parse_dates, avec un s
ventes_bug.info()                                           # info est une méthode : il faut les parenthèses
ventes_bug["statut"].value_counts()                         # les noms de colonnes sont sensibles à la casse
```

- `parse_date` lève `TypeError: read_csv() got an unexpected keyword argument 'parse_date'. Did you mean 'parse_dates'?` : lis bien la fin du message, Python te donne la réponse.
- `ventes_bug.info` sans parenthèses ne lève **aucune erreur** : il affiche juste la description de la méthode, sans l'exécuter.
- `"Statut"` lève `KeyError: 'Statut'` : la colonne s'appelle `statut`, en minuscules.
</details>